In [ ]:
"""
Borough name-matching check between gapscore_v2.csv and boundary geojson.
Run this BEFORE loading anything into Tableau to catch mismatches early.
"""

import pandas as pd
import geopandas as gpd

# ---- 1. Load your data ----
gap=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore_v2.csv")

# Adjust path to wherever you save the downloaded boundary file
gdf=gpd.read_file(r"C:\Users\Hp\Downloads\Project 2026 DS\boundaryfile.geojson")

# ---- 2. Your 33 London boroughs (adjust if your existing list differs) ----
london_boroughs=[
    "Barking and Dagenham","Barnet","Bexley","Brent","Bromley","Camden",
    "Croydon","Ealing","Enfield","Greenwich","Hackney",
    "Hammersmith and Fulham","Haringey","Harrow","Havering","Hillingdon",
    "Hounslow","Islington","Kensington and Chelsea","Kingston upon Thames",
    "Lambeth","Lewisham","Merton","Newham","Redbridge",
    "Richmond upon Thames","Southwark","Sutton","Tower Hamlets",
    "Waltham Forest","Wandsworth","Westminster","City of London"
]

# ---- 3. Find the name column in the geojson (usually LAD24NM or similar) ----
print("GeoDataFrame columns:",gdf.columns.tolist())
name_col="LAD24NM"  # change this if your file uses a different column name

# ---- 4. Filter geojson down to London boroughs only ----
gdf_london=gdf[gdf[name_col].isin(london_boroughs)].copy()
print(f"\nMatched {len(gdf_london)} / 33 boroughs by direct name match in geojson.")

# ---- 5. Cross-check sets ----
gap_names=set(gap['borough'].unique())
geo_names=set(gdf_london[name_col].unique())
expected=set(london_boroughs)

print("\n--- In gapscore_v2.csv but NOT in filtered geojson ---")
print(gap_names - geo_names or "None — all matched")

print("\n--- In filtered geojson but NOT in gapscore_v2.csv ---")
print(geo_names - gap_names or "None — all matched")

print("\n--- Expected 33 boroughs missing from geojson entirely ---")
print(expected - set(gdf[name_col].unique()) or "None — all 33 present in raw geojson")

print("\n--- Expected 33 boroughs missing from gapscore_v2.csv ---")
print(expected - gap_names or "None — all 33 present in gapscore")

# ---- 6. If mismatches found, inspect raw geojson names for near-matches ----
if gap_names - geo_names:
    print("\n--- Checking raw geojson for near-matches (case/spacing/hyphen issues) ---")
    all_geo_names=gdf[name_col].unique()
    for missing in (gap_names - geo_names):
        close=[n for n in all_geo_names if missing.lower().replace("-"," ").strip()
                  in n.lower().replace("-"," ").strip()
                  or n.lower().replace("-"," ").strip()
                  in missing.lower().replace("-"," ").strip()]
        print(f"  '{missing}' -> possible matches in geojson: {close}")

# ---- 7. Save the filtered, clean London-only boundary file ----
if len(gdf_london) == 33 and not (gap_names - geo_names) and not (geo_names - gap_names):
    out_path=r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\london_boundaries_clean.geojson"
    gdf_london.to_file(out_path,driver="GeoJSON")
    print(f"\nAll names matched cleanly. Saved filtered file to:\n{out_path}")
else:
    print("\nMismatches found above — fix names in gapscore_v2.csv or apply a rename "
          "dict before saving the filtered geojson. Do NOT load into Tableau yet.")

In [ ]:
import os
import datetime

files_to_check=[
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore_v2.csv",
    r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores.csv",
    r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_full33.csv",
    r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv",
]

for f in files_to_check:
    if os.path.exists(f):
        mtime=os.path.getmtime(f)
        print(f"{f}\n  last modified: {datetime.datetime.fromtimestamp(mtime)}\n")
    else:
        print(f"{f}\n  NOT FOUND\n")

In [ ]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Hp\Downloads\Project 2026 DS\GAP_data\gap_scores_equity.csv")
print(df.columns.tolist())
print(df.shape)
print(df[['borough']].nunique())  # should be 33
print(df.isnull().sum())  # check for unexpected gaps, especially in classification/equity_gap columns
df.head(10)